# Final LSTM — Fine-tuned Prof-Style (Bi-directional + Grouped Split)

**Based on:** `lstm_prof_style.ipynb` (faithful mirror of the professor's
Keras architecture).

**What changed and why:**

| Setting | Prof-style | This notebook | Reason |
|---|---|---|---|
| Split | random 60/20/20 | **tile-grouped** T02CN* → T03CWT | no data leakage |
| LSTM direction | uni-directional | **bi-directional** | richer along-track context |
| SEQ_LEN | 5 | **9** | 4-before + center + 4-after |
| hidden / head | 48 / 16 | **64 / 32** | slightly more capacity |
| LR schedule | none | **ReduceLROnPlateau** | adapts when val stalls |
| Grad clipping | none | **max_norm=1.0** | stabilises Bi-LSTM |
| Early stop | none (EPOCHS=50) | **patience=12** | saves the best epoch |
| Test checkpoint | final.pt | **best.pt** | honest best-by-val |

Architecture:

```
Bi-LSTM(64, tanh)  [2 × 64 = 128-dim output]
Dropout(0.4)
Linear(128, 32)  ELU  Dropout(0.4)
Linear(32,  32)  ELU  Dropout(0.4)
Linear(32,   3)
```


## 0. Setup

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")


In [ ]:
import json, math, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

print("torch:", torch.__version__,
      "cuda:", torch.cuda.is_available(),
      "device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


## 1. Config

In [ ]:
PROJECT_ROOT = Path("/home/spant/Research Seminar/Project")
EXP_NAME     = "lstm_final_v1"

CSV_DIR  = PROJECT_ROOT / "IS2_Corrected_data"
RUN_DIR  = PROJECT_ROOT / "runs" / EXP_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

# ---- data ----
SEED          = 42
NUM_CLASSES   = 3
SEQ_LEN       = 9          # 4 before + center + 4 after  (was 5)
NEARBY        = SEQ_LEN // 2
GROUPED_SPLIT = True       # always tile-grouped for honest evaluation

# ---- training ----
BATCH_SIZE    = 32
EPOCHS        = 80
LR            = 8.886176350890356e-4   # same LR as prof
WEIGHT_DECAY  = 1e-4                   # small L2 (prof had 0, this helps Bi-LSTM)
PATIENCE      = 12                     # real early stopping
NUM_WORKERS   = 2

# ---- architecture ----
LSTM_HIDDEN   = 64         # was 48; doubled output because bidirectional
LSTM_DROPOUT  = 0.4
HEAD_HIDDEN   = 32         # was 16
HEAD_DROPOUT  = 0.4
GRAD_CLIP     = 1.0        # gradient clipping max norm

# ---- focal loss (same as prof's final compile cell) ----
ALPHA = [0.05, 0.45, 0.60]
GAMMA = 2.0

CLASS_NAMES      = ["ice", "thin_ice", "water"]
CLASS_NAMES_DISP = ["thick ice", "thin ice", "water"]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("Run dir:", RUN_DIR)
print(f"SEQ_LEN={SEQ_LEN}  hidden={LSTM_HIDDEN}  bidir=True  clip={GRAD_CLIP}")
print(f"Focal alpha={ALPHA}  gamma={GAMMA}")


## 2. Load 10 m segment CSVs

Same 8 features as the prof-style notebook.  `h_diff` and
`rel_height_min_elev` are derived the same way.

In [ ]:
FEATURES = [
    "h_cor_mean", "h_diff", "rel_height_min_elev", "height_sd",
    "pcnth_mean", "pcnt_mean", "bcnt_mean", "brate_mean",
]
N_FEATS = len(FEATURES)


def load_segment_csv(csv_path: Path):
    df = pd.read_csv(csv_path)
    needed = ["h_cor_mean", "h_cor_med", "x_atc", "label",
              "height_sd", "pcnth_mean", "pcnt_mean", "bcnt_mean", "brate_mean"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"{csv_path.name}: missing columns {missing}")
    df = df[df["label"].isin([0, 1, 2])].copy()
    df = df.dropna(subset=needed)
    df = df.sort_values("x_atc").reset_index(drop=True)
    df["h_diff"]              = df["h_cor_mean"] - df["h_cor_med"]
    df["rel_height_min_elev"] = df["h_cor_mean"] - df["h_cor_mean"].min()
    return df


csv_files = sorted(CSV_DIR.glob("ATL03_*_done.csv"))
print(f"found {len(csv_files)} CSVs")

records = []
for p in csv_files:
    parts = p.stem.split("_")
    tile  = parts[3]; beam = parts[4]
    seg   = load_segment_csv(p)
    seg["tile"] = tile; seg["beam"] = beam; seg["src"] = p.name
    records.append((tile, beam, seg))
    print(f"  {p.name}: {len(seg):,} segs  labels={seg['label'].value_counts().to_dict()}")
print(f"total: {sum(len(s) for _, _, s in records):,} segments")


## 3. Build (N, SEQ_LEN, 8) sliding windows

Wider window than the prof (9 steps instead of 5).

In [ ]:
def sliding_windows(seg_df, seq_len=SEQ_LEN, nearby=NEARBY):
    arr = seg_df[FEATURES].to_numpy(dtype=np.float32)
    lab = seg_df["label"].to_numpy(dtype=np.int64)
    if len(arr) < seq_len:
        return (np.empty((0, seq_len, N_FEATS), dtype=np.float32),
                np.empty((0,), dtype=np.int64))
    n = len(arr) - 2 * nearby
    X = np.zeros((n, seq_len, N_FEATS), dtype=np.float32)
    for k in range(seq_len):
        X[:, k, :] = arr[k : k + n]
    y = lab[nearby : nearby + n]
    valid = (y >= 0) & (y < NUM_CLASSES)
    return X[valid], y[valid]


bundles = []
for tile, beam, seg in records:
    X, y = sliding_windows(seg)
    if len(X) == 0:
        continue
    bundles.append({"tile": tile, "beam": beam, "X": X, "y": y})

total_n = sum(len(b["y"]) for b in bundles)
print(f"{len(bundles)} tracks, {total_n:,} samples total")


## 4. Train / val / test split (grouped by tile)

Train tiles: `T02CNA + T02CNC`.  Test tile: `T03CWT` (new region, new date).
Val = 10 % random subset of train tiles.

In [ ]:
train_tiles = {"T02CNA", "T02CNC"}
test_tiles  = {"T03CWT"}

X_tr = np.concatenate([b["X"] for b in bundles if b["tile"] in train_tiles])
y_tr = np.concatenate([b["y"] for b in bundles if b["tile"] in train_tiles])
X_te = np.concatenate([b["X"] for b in bundles if b["tile"] in test_tiles])
y_te = np.concatenate([b["y"] for b in bundles if b["tile"] in test_tiles])

rng  = np.random.RandomState(SEED)
idx  = rng.permutation(len(X_tr))
cut  = int(0.10 * len(idx))
val_idx, tr_idx = idx[:cut], idx[cut:]
X_val, y_val = X_tr[val_idx], y_tr[val_idx]
X_tr,  y_tr  = X_tr[tr_idx],  y_tr[tr_idx]

print(f"train: {len(X_tr):,}   val: {len(X_val):,}   test: {len(X_te):,}")
for name, y in [("train", y_tr), ("val", y_val), ("test", y_te)]:
    c = np.bincount(y, minlength=NUM_CLASSES)
    p = c / max(c.sum(), 1) * 100
    print(f"  {name}: " + "  ".join(f"{n}={v} ({pv:.1f}%)"
          for n, v, pv in zip(CLASS_NAMES, c, p)))


## 5. Standardize features (z-score on train set)

In [ ]:
flat_tr = X_tr.reshape(-1, N_FEATS)
means   = flat_tr.mean(axis=0)
stds    = flat_tr.std(axis=0) + 1e-6


def standardize(X):
    return (X - means[None, None, :]) / stds[None, None, :]


X_tr  = standardize(X_tr).astype(np.float32)
X_val = standardize(X_val).astype(np.float32)
X_te  = standardize(X_te).astype(np.float32)
print("means:", means.round(3))
print("stds: ", stds.round(3))


## 6. Dataset + DataLoaders

In [ ]:
class SegmentDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).long()

    def __len__(self): return len(self.y)

    def __getitem__(self, i): return self.X[i], self.y[i]


train_loader = DataLoader(SegmentDataset(X_tr,  y_tr),  batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, drop_last=True)
val_loader   = DataLoader(SegmentDataset(X_val, y_val), batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(SegmentDataset(X_te,  y_te),  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)
print(f"train steps: {len(train_loader)}  val: {len(val_loader)}  test: {len(test_loader)}")


## 7. Model — Bi-LSTM(64) + Dense(32, elu) × 2 + Dense(3)

`bidirectional=True` doubles the output dimension to 128.
The head is slightly wider than the prof's (32 vs 16).

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class FinetunedLSTM(nn.Module):
    def __init__(self, n_features=N_FEATS, hidden=LSTM_HIDDEN,
                 head_hidden=HEAD_HIDDEN, head_dropout=HEAD_DROPOUT,
                 lstm_dropout=LSTM_DROPOUT, num_classes=NUM_CLASSES):
        super().__init__()
        # bidirectional=True → output dim = hidden * 2
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden,
                            num_layers=1, batch_first=True,
                            bidirectional=True)
        out_dim = hidden * 2
        self.head = nn.Sequential(
            nn.Dropout(lstm_dropout),
            nn.Linear(out_dim, head_hidden), nn.ELU(inplace=True),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden, head_hidden), nn.ELU(inplace=True),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden, num_classes),
        )

    def forward(self, x):
        out, _ = self.lstm(x)           # (B, T, hidden*2)
        # mean-pool over the full window for a richer representation
        feat = out.mean(dim=1)          # (B, hidden*2)
        return self.head(feat)          # (B, num_classes)


model = FinetunedLSTM().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"FinetunedLSTM parameters: {n_params:,}")
print(model)


## 8. Focal Loss + Adam + ReduceLROnPlateau

Same focal-loss formula as the prof-style notebook.
Added `ReduceLROnPlateau`: if val mIoU does not improve for 5 epochs
the LR is halved (down to a floor of 1e-6).

In [ ]:
class CategoricalFocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.register_buffer("alpha", torch.tensor(alpha, dtype=torch.float32))
        self.gamma = float(gamma)

    def forward(self, logits, target):
        log_probs = F.log_softmax(logits, dim=1)
        log_p_t   = log_probs.gather(1, target.unsqueeze(1)).squeeze(1)
        p_t       = log_p_t.exp().clamp(min=1e-8, max=1.0 - 1e-8)
        alpha_t   = self.alpha.to(logits.device)[target]
        per_sample = -alpha_t * (1.0 - p_t).pow(self.gamma) * log_p_t
        return per_sample.mean()


criterion = CategoricalFocalLoss(alpha=ALPHA, gamma=GAMMA).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5, min_lr=1e-6, verbose=True)
print("loss + optimizer + scheduler ready")


## 9. Training loop (with grad clipping + early stopping)

In [ ]:
def cm_accum(cm, logits, targets):
    pred = logits.argmax(1).detach().cpu().numpy().ravel()
    t    = targets.detach().cpu().numpy().ravel()
    idx  = NUM_CLASSES * t + pred
    cm  += np.bincount(idx, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)


def metrics_from_cm(cm):
    iou = []
    for c in range(NUM_CLASSES):
        tp = cm[c, c]; fp = cm[:, c].sum() - tp; fn = cm[c, :].sum() - tp
        denom = tp + fp + fn
        iou.append(float(tp / denom) if denom > 0 else 0.0)
    iou = np.array(iou, dtype=np.float64)
    pix_acc = float(np.diag(cm).sum() / max(cm.sum(), 1))
    return float(iou.mean()), iou, pix_acc


best_val      = -1.0
patience_left = PATIENCE
log           = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()

    # ---- train ----
    model.train()
    tr_loss = 0.0; n_seen = 0
    tr_cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for X, y in train_loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        tr_loss += loss.item() * X.size(0); n_seen += X.size(0)
        cm_accum(tr_cm, logits, y)
    tr_loss /= max(n_seen, 1)
    tr_miou, _, tr_acc = metrics_from_cm(tr_cm)

    # ---- val ----
    model.eval()
    va_loss = 0.0; n_seen = 0
    va_cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            va_loss += criterion(model(X), y).item() * X.size(0)
            n_seen  += X.size(0)
            cm_accum(va_cm, model(X), y)
    va_loss /= max(n_seen, 1)
    va_miou, va_iou, va_acc = metrics_from_cm(va_cm)

    scheduler.step(va_miou)
    current_lr = optimizer.param_groups[0]["lr"]

    log.append({"epoch": epoch,
                "train_loss": tr_loss, "train_miou": tr_miou, "train_acc": tr_acc,
                "val_loss":   va_loss, "val_miou":   va_miou, "val_acc":   va_acc,
                "val_ice": va_iou[0], "val_thin": va_iou[1], "val_water": va_iou[2],
                "lr": current_lr})
    print(f"ep {epoch:03d}  tr_loss {tr_loss:.4f}  tr_mIoU {tr_miou:.4f}  |  "
          f"va_loss {va_loss:.4f}  va_mIoU {va_miou:.4f}  "
          f"[{va_iou.round(3).tolist()}]  lr={current_lr:.2e}  "
          f"({time.perf_counter()-t0:.0f}s)")

    if va_miou > best_val + 1e-4:
        best_val      = va_miou
        patience_left = PATIENCE
        torch.save({"epoch": epoch, "model_state": model.state_dict(),
                    "alpha": ALPHA, "gamma": GAMMA,
                    "means": means, "stds": stds,
                    "val_metrics": {"miou": va_miou, "per_iou": va_iou.tolist(),
                                    "pix_acc": va_acc, "loss": va_loss}},
                   RUN_DIR / "best.pt")
    else:
        patience_left -= 1
        if patience_left <= 0:
            print(f"Early stop at epoch {epoch}. Best val mIoU = {best_val:.4f}")
            break

pd.DataFrame(log).to_csv(RUN_DIR / "metrics.csv", index=False)
print(f"\nBest val mIoU: {best_val:.4f}  saved → best.pt")


## 10. Test evaluation (best.pt)

In [ ]:
ck = torch.load(RUN_DIR / "best.pt", map_location=device, weights_only=False)
model.load_state_dict(ck["model_state"]); model.eval()
print(f"Loaded epoch {ck['epoch']}  (val mIoU {ck['val_metrics']['miou']:.4f})")

test_cm   = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
test_loss = 0.0; n_seen = 0
with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits     = model(X)
        test_loss += criterion(logits, y).item() * X.size(0)
        n_seen    += X.size(0)
        cm_accum(test_cm, logits, y)
test_loss /= max(n_seen, 1)
test_miou, test_iou, test_acc = metrics_from_cm(test_cm)

# Per-class precision / recall / F1
prec = np.zeros(NUM_CLASSES); rec = np.zeros(NUM_CLASSES); f1 = np.zeros(NUM_CLASSES)
for c in range(NUM_CLASSES):
    tp = test_cm[c, c]; fp = test_cm[:, c].sum() - tp; fn = test_cm[c, :].sum() - tp
    prec[c] = tp / (tp + fp)   if (tp + fp) else 0.0
    rec[c]  = tp / (tp + fn)   if (tp + fn) else 0.0
    f1[c]   = 2*prec[c]*rec[c] / (prec[c]+rec[c]) if (prec[c]+rec[c]) else 0.0

print(f"\nTEST  mIoU {test_miou:.4f}  pix_acc {test_acc:.4f}  loss {test_loss:.4f}")
print(f"per-class IoU       : {test_iou.round(4).tolist()}")
print(f"per-class precision : {prec.round(4).tolist()}")
print(f"per-class recall    : {rec.round(4).tolist()}")
print(f"per-class F1        : {f1.round(4).tolist()}")
print(f"macro-F1            : {f1.mean():.4f}")

with open(RUN_DIR / "test_metrics.json", "w") as f:
    json.dump({"miou": test_miou, "per_iou": test_iou.tolist(),
               "pix_acc": test_acc, "loss": test_loss,
               "precision": prec.tolist(), "recall": rec.tolist(),
               "f1": f1.tolist(), "macro_f1": float(f1.mean()),
               "alpha": ALPHA, "gamma": GAMMA,
               "grouped_split": GROUPED_SPLIT}, f, indent=2)


## 11. Confusion matrix (prof-style percentage view)

In [ ]:
def plot_cm_percent(cm, title, save_path):
    pct = cm.astype(np.float64)
    row_sums = pct.sum(axis=1, keepdims=True)
    pct = np.where(row_sums > 0, pct / np.maximum(row_sums, 1) * 100.0, 0.0)
    fig, ax = plt.subplots(figsize=(7.2, 5.8))
    im = ax.imshow(pct, cmap="Blues", vmin=0, vmax=100, aspect="auto")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels([f"Predicted {c}" for c in CLASS_NAMES_DISP], fontsize=11)
    ax.set_yticklabels(CLASS_NAMES_DISP, fontsize=11)
    ax.set_xlabel("Predicted", fontsize=12); ax.set_ylabel("Actual", fontsize=12)
    ax.set_title(title, fontsize=13)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            v = pct[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if v > 55 else "black", fontsize=13)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()


plot_cm_percent(test_cm, "Confusion Matrix (Percentages) — Final LSTM",
                RUN_DIR / "confmat.png")
plot_cm_percent(test_cm,
                f"Confusion Matrix  alpha={ALPHA}  gamma={GAMMA}  SEQ={SEQ_LEN}  Bi-LSTM",
                RUN_DIR / f"confmat_bidir_seq{SEQ_LEN}.png")


## 12. Loss + mIoU curves

In [ ]:
hist = pd.read_csv(RUN_DIR / "metrics.csv")
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(hist["epoch"], hist["train_loss"], label="train")
axes[0].plot(hist["epoch"], hist["val_loss"],   label="val")
axes[0].set_title("Focal loss"); axes[0].set_xlabel("epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist["epoch"], hist["train_miou"], label="train")
axes[1].plot(hist["epoch"], hist["val_miou"],   label="val")
axes[1].plot(hist["epoch"], hist["val_thin"],   "--", label="val thin_ice", alpha=0.7)
axes[1].set_title("mIoU"); axes[1].set_xlabel("epoch")
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(hist["epoch"], hist["lr"])
axes[2].set_title("Learning rate"); axes[2].set_xlabel("epoch")
axes[2].set_yscale("log"); axes[2].grid(alpha=0.3, which="both")

plt.tight_layout()
plt.savefig(RUN_DIR / "loss_curve.png", dpi=160, bbox_inches="tight")
plt.show()


## 13. Done

Files in `runs/lstm_final_v1/`:

| file | what it is |
|---|---|
| `best.pt` | best-by-val checkpoint (includes feature mean/std) |
| `metrics.csv` | per-epoch train/val curves + LR |
| `test_metrics.json` | test mIoU, pix_acc, per-class P/R/F1/IoU, macro-F1 |
| `confmat.png` | prof-style percentage confusion matrix |
| `loss_curve.png` | loss + mIoU + LR training diagnostics |

**Comparison to prof-style:**

| | lstm_prof_style | **final (this notebook)** |
|---|---|---|
| direction | uni | **bi** |
| SEQ_LEN | 5 | **9** |
| split | random | **tile-grouped** |
| early stop | none | **patience=12** |
| LR decay | none | **ReduceLROnPlateau** |
